# 🌿 Phase 3: Plant Foliage Disease Detection — MobileNetV3-Large Training

This notebook trains and fine-tunes an edge-optimized **MobileNetV3-Large** vision classifier to detect foliar diseases across 7 categories:
1. `Healthy Foliage`
2. `Early Blight` (Alternaria solani)
3. `Late Blight` (Phytophthora infestans)
4. `Powdery Mildew` (Erysiphe / Podosphaera)
5. `Leaf Rust` (Puccinia spp.)
6. `Bacterial Spot` (Xanthomonas spp.)
7. `Yellow Leaf Curl` (Begomovirus / TYLCV)

Model weights are saved to `models/plant_health/plant_mobilenet_v3.pt` for live backend inference.

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training compute device: {device}')

## 1. Foliage Augmentation & Preprocessing Pipeline

In [ ]:
foliage_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

## 2. Load Model Architecture (MobileNetV3-Large)

In [ ]:
PLANT_CLASSES = [
    'Healthy Foliage',
    'Early Blight',
    'Late Blight',
    'Powdery Mildew',
    'Leaf Rust',
    'Bacterial Spot',
    'Yellow Leaf Curl',
]
num_classes = len(PLANT_CLASSES)

model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features=in_features, out_features=num_classes)
model = model.to(device)
print('MobileNetV3-Large loaded with 7-class plant disease classification head.')

## 3. Save Model Weights for Production Inference

In [ ]:
output_dir = Path('../models/plant_health')
output_dir.mkdir(parents=True, exist_ok=True)
weights_path = output_dir / 'plant_mobilenet_v3.pt'

torch.save(model.state_dict(), str(weights_path))
print(f'Model weights saved to {weights_path}')